Step 1 - Setup 

In [15]:
import torch
import torch.nn as nn
from torchvision import models, transforms
import cv2
from PIL import Image
import numpy as np
import os
import gradio as gr
gr.close_all()
from huggingface_hub import InferenceClient
from dotenv import load_dotenv

load_dotenv("D:/Deepfake-detection/.env")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.efficientnet_b0(weights=None)
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 2)
model.load_state_dict(torch.load("D:/Deepfake-detection/saved_models/efficientnet_best.pth", map_location=device))
model = model.to(device)
model.eval()
class_names = ['fake', 'real']
IMG_SIZE = 224
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

HF_TOKEN = os.getenv("HF_TOKEN")
client = InferenceClient(api_key=HF_TOKEN, provider="auto")
print("All models and clients loaded.")

Closing server running on port: 7860
All models and clients loaded.


Step 2 - The three prediction functions

plit into small reusable helper functions (crop_face_from_frame, classify_face_crop) so image/video/webcam all share the same core logic instead of duplicating it three times — cleaner code

Webcam tab intentionally skips the GenAI explanation call — since webcam frames update rapidly, calling the Hugging Face API on every single frame would be slow and could hit rate limits fast; it just shows a quick label instead

In [12]:
def crop_face_from_frame(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
    if len(faces) > 0:
        faces_sorted = sorted(faces, key=lambda f: f[2] * f[3], reverse=True)
        x, y, w, h = faces_sorted[0]
        return frame[y:y+h, x:x+w]
    return frame

def classify_face_crop(face_crop):
    face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(face_rgb)
    input_tensor = transform(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)[0]
    return probs.cpu().numpy()


def generate_explanation(prediction, confidence, fake_prob, real_prob):
    prompt = f"""Detection result: {prediction.upper()}
Confidence: {confidence}%
Fake probability: {fake_prob}%
Real probability: {real_prob}%

Write a short, 2-3 sentence explanation for the user about this deepfake
detection result. Be honest this is a statistical prediction, not certainty."""
    try:
        response = client.chat.completions.create(
            model="deepseek-ai/DeepSeek-V3-0324",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=150
        )
        return response.choices[0].message.content
    except Exception as e:
        print("EXPLANATION ERROR:", type(e).__name__, str(e))  # temporary debug line
        return "(Explanation unavailable right now.)"
# ---- IMAGE TAB ----
def image_predict(pil_image):
    if pil_image is None:
        return "Please upload an image."
    img_array = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
    face_crop = crop_face_from_frame(img_array)
    probs = classify_face_crop(face_crop)
    pred_idx = int(np.argmax(probs))
    prediction = class_names[pred_idx]
    confidence = round(float(probs[pred_idx]) * 100, 2)
    fake_prob = round(float(probs[0]) * 100, 2)
    real_prob = round(float(probs[1]) * 100, 2)
    explanation = generate_explanation(prediction, confidence, fake_prob, real_prob)
    return (f"Prediction: {prediction.upper()}\nConfidence: {confidence}%\n\n"
            f"Fake: {fake_prob}% | Real: {real_prob}%\n\nExplanation:\n{explanation}")

# ---- VIDEO TAB ----
def video_predict(video_path):
    if video_path is None:
        return "Please upload a video."
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return "Could not read video."
    
    num_frames = 5
    frame_indices = [int(i * total_frames / num_frames) for i in range(num_frames)]
    all_probs = []
    
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        success, frame = cap.read()
        if not success:
            continue
        face_crop = crop_face_from_frame(frame)
        probs = classify_face_crop(face_crop)
        all_probs.append(probs)
    cap.release()
    
    if len(all_probs) == 0:
        return "No frames could be processed."
    
    avg_probs = np.mean(all_probs, axis=0)
    pred_idx = int(np.argmax(avg_probs))
    prediction = class_names[pred_idx]
    confidence = round(float(avg_probs[pred_idx]) * 100, 2)
    fake_prob = round(float(avg_probs[0]) * 100, 2)
    real_prob = round(float(avg_probs[1]) * 100, 2)
    explanation = generate_explanation(prediction, confidence, fake_prob, real_prob)
    return (f"Prediction: {prediction.upper()}\nConfidence: {confidence}%\n"
            f"(Averaged across {len(all_probs)} sampled frames)\n\n"
            f"Fake: {fake_prob}% | Real: {real_prob}%\n\nExplanation:\n{explanation}")

# ---- WEBCAM TAB (single frame snapshot from Gradio's webcam widget) ----
# Global variable to hold the most recent webcam frame
latest_webcam_frame = {"frame": None}

def store_latest_frame(pil_image):
    latest_webcam_frame["frame"] = pil_image
    return None  # streaming function doesn't need to return anything visible

def take_snapshot_and_analyze(pil_image):
    if pil_image is None:
        return "No frame captured — click the camera icon on the video feed first, then click Analyze."
    
    img_array = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
    
    max_dim = 480
    h, w = img_array.shape[:2]
    if max(h, w) > max_dim:
        scale = max_dim / max(h, w)
        img_array = cv2.resize(img_array, (int(w * scale), int(h * scale)))
    
    face_crop = crop_face_from_frame(img_array)
    probs = classify_face_crop(face_crop)
    pred_idx = int(np.argmax(probs))
    prediction = class_names[pred_idx]
    confidence = round(float(probs[pred_idx]) * 100, 2)
    return f"Prediction: {prediction.upper()} ({confidence}%)"
import time

def webcam_predict(pil_image):
    if pil_image is None:
        return "Waiting for webcam frame..."
    
    t0 = time.time()
    img_array = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
    
    max_dim = 480
    h, w = img_array.shape[:2]
    if max(h, w) > max_dim:
        scale = max_dim / max(h, w)
        img_array = cv2.resize(img_array, (int(w * scale), int(h * scale)))
    
    t1 = time.time()
    face_crop = crop_face_from_frame(img_array)
    t2 = time.time()
    probs = classify_face_crop(face_crop)
    t3 = time.time()
    
    print(f"Resize: {t1-t0:.2f}s | Face detect: {t2-t1:.2f}s | Classify: {t3-t2:.2f}s")
    
    pred_idx = int(np.argmax(probs))
    prediction = class_names[pred_idx]
    confidence = round(float(probs[pred_idx]) * 100, 2)
    return f"Prediction: {prediction.upper()} ({confidence}%)"

Step 3 - Build the tabbed interface

gr.Blocks(...) — the flexible layout container (vs. the simpler gr.Interface we used before)

gr.Tab(...) — creates separate tabs, each with its own inputs/outputs/logic

gr.Video(...) — Gradio's built-in video upload widget, handles the file automatically

sources=["webcam"] on the image input — tells Gradio to show a "take snapshot from webcam" option instead of just file upload

Each tab has its own Analyze button rather than auto-running on upload — gives the user control over when prediction runs (important for video especially, since it takes a moment to process)

In [13]:
with gr.Blocks(title="AI-Powered Deepfake Detection System") as demo:
    gr.Markdown("# 🎭 AI-Powered Deepfake Detection System")
    gr.Markdown("Detect deepfakes in images, videos, or live webcam feed — powered by EfficientNet-B0.")
    
    with gr.Tab("📷 Image"):
        img_input = gr.Image(type="pil", label="Upload an image")
        img_button = gr.Button("Analyze Image")
        img_output = gr.Textbox(label="Result", lines=10)
        img_button.click(fn=image_predict, inputs=img_input, outputs=img_output)
    
    with gr.Tab("🎥 Video"):
        vid_input = gr.Video(label="Upload a video")
        vid_button = gr.Button("Analyze Video")
        vid_output = gr.Textbox(label="Result", lines=10)
        vid_button.click(fn=video_predict, inputs=vid_input, outputs=vid_output)
    
    with gr.Tab("📹 Webcam"):
        gr.Markdown("Click the camera icon on the video feed below to capture a photo, then click Analyze.")
        webcam_input = gr.Image(type="pil", label="Webcam", sources=["webcam"])
        
        with gr.Row():
            snapshot_button = gr.Button("📸 Analyze Snapshot", variant="primary")
            stop_button = gr.Button("🛑 Stop Camera", variant="stop")
        
        webcam_output = gr.Textbox(label="Result")
        
        snapshot_button.click(fn=take_snapshot_and_analyze, inputs=webcam_input, outputs=webcam_output)
        stop_button.click(fn=lambda: None, inputs=None, outputs=webcam_input)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [17]:
client = InferenceClient(api_key=HF_TOKEN, provider="auto")
print("Client recreated.")

Client recreated.


In [18]:
try:
    response = client.chat.completions.create(
        model="deepseek-ai/DeepSeek-V3-0324",
        messages=[{"role": "user", "content": "Say hello in one sentence."}],
        max_tokens=50
    )
    print(response.choices[0].message.content)
except Exception as e:
    print("ERROR:", type(e).__name__, str(e))

"Hello!" 😊


In [19]:
test = generate_explanation("fake", 65.36, 65.36, 34.64)
print(test)

This result suggests the content is likely a deepfake, with a 65.4% probability of being manipulated. However, this is a statistical estimate, not a definitive verdict—there's still a 34.6% chance it could be real. For critical decisions, consider additional verification or expert review.


In [20]:
model.load_state_dict(torch.load("D:/Deepfake-detection/saved_models/efficientnet_finetuned.pth", map_location=device))
model.eval()
print("Loaded fine-tuned model.")

Loaded fine-tuned model.
